In [ ]:
# --- 1. CORE LIBRARIES ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 2. SKLEARN: PREPROCESSING & SPLITTING ---
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- 3. SKLEARN: MODELS (Covers Regression & Classification) ---
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier

# --- 4. SKLEARN: METRICS ---
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)




# Configuration for cleaner output
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None) # Show all columns
print("Libraries Imported Successfully")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
#task 1
path = os.path.join(path,'/kaggle/input/q1-ka-ai-2026/Q1_data.csv')
df = pd.read_csv(path)

print(f"Dataset shape: {df.shape}")


In [ ]:
# Task 2:
df.head()

In [ ]:
# Task 3:
df.info()

In [ ]:
# Task 4:
df.describe()

In [ ]:
# Task 5:
# distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.show()



In [ ]:
# Task 1:
df.drop(columns=['Order_ID']) # id - names rows can misslead the model and they are not usefull for training

In [ ]:
# Task 2:

df = df.dropna(subset=['Delivery_Time']) #droping missing values in the target coulmn

#how many data are missing from each coulmn
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

# Define Columns by Type
# We need to treat numbers and text differently
num_cols = df.select_dtypes(include=['number']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# Create Imputers
# Strategy: 'median' for numbers (robust to outliers), 'most_frequent' for text
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

# --- 3. Apply Imputation ---
# We use .copy() to keep original df safe
df_clean = df.copy()

# Fill Missing Values
df_clean[num_cols] = num_imputer.fit_transform(df[num_cols])
df_clean[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# Verify Cleaning
print(f"Missing values after cleaning: {df_clean.isnull().sum().sum()}")


In [ ]:
# Task 3:
def check_duplicates(df):
  duplicates = df_clean.duplicated().sum() #sum the number of duplicate # checks the row (data sample )
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True) #delete them
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4:
# --- 1. Separate Features (X) and Target (y) ---
X = df_clean.drop(columns=['Delivery_Time'])
y = df_clean['Delivery_Time']

# --- 2. Split Data ---
# Logic: If Classification -> Use Stratify. If Regression -> No Stratify.
is_classification = (y.dtype == 'object') or (y.nunique() < 20)

if is_classification:
    print("Detected Classification Task -> Using Stratified Split")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
else:
    print("Detected Regression Task -> Using Random Split")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

print(f"Train Shape: {X_train.shape}")
print(f"Test Shape:  {X_test.shape}")

In [ ]:
# Task 5:
# --- 1. Update Column Lists (In case Features Changed) ---
num_features = X_train.select_dtypes(include=['number']).columns
cat_features = X_train.select_dtypes(include=['object', 'category']).columns

# --- 2. Define Transformers ---
# Numerical: Standardize (Z-score)
scaler = StandardScaler()

# Categorical: One-Hot Encode (handle_unknown='ignore' prevents errors on new categories)
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# --- 3. Build Preprocessor ---
preprocessor = ColumnTransformer(
    transformers=[
        ('num', scaler, num_features),
        ('cat', encoder, cat_features)
    ],
    verbose_feature_names_out=False # Keeps column names clean
)

# --- 4. Apply Transformations ---
# Fit ONLY on Train, Transform on both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert back to DataFrame for readability (Optional, but helpful for debugging)
feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

print("Data Processed & Ready for Modeling")
display(X_train_df.head(3))

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1:

X = df_clean.drop(columns=['Delivery_Time'])
y = df_clean['Delivery_Time']




In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import train_test_split, KFold

fkfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
   # Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
   model.fit(X_train_scaled, y_train)
   print("Model trained!")

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")

In [ ]:
# --- Extract Feature Importance (Random Forest Only) ---

# 1. Get the model from the pipeline
# (Assuming your pipeline step is named 'model')
rf_model = my_pipeline.named_steps['model']

# 2. Get importance scores
importances = rf_model.feature_importances_

# 3. Get feature names
# (If you used OneHotEncoder, this is tricky. This is a simplified safe version)
# We will use the numerical columns + generic names for categorical to avoid errors
feature_names = num_cols.tolist() + [f"Cat_{i}" for i in range(len(importances) - len(num_cols))]

# 4. Create a DataFrame
feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).head(10) # Top 10

# 5. Plot
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df, palette='magma')
plt.title("Top 10 Most Important Features")
plt.show()

print("💡 Analysis: The top feature is the biggest driver of the decision.")

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: